# APM_50441_EP - Machine Learning 2 - Lab 4: Unsupervised learning, dimension reduction, clustering and generative models

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/APM-50441-EP/APM_50441_EP-2026-2027/blob/main/PC4/APM_50441_EP_Lab_4_simplified.ipynb) or run it on your own computer (see *Working environment* below).

In [ ]:
# On Colab (R runtime) only: installs the packages of this lab (once per session, a few minutes).
# On your own computer, use PC/env/install_packages.R instead (see Working environment).
if (nzchar(Sys.getenv("COLAB_RELEASE_TAG"))) {
  options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))  # binary packages
  source("https://raw.githubusercontent.com/APM-50441-EP/APM_50441_EP-2026-2027/main/env/install_packages.R")
  install_lab("lab4")
}

This lab illustrates the fourth lecture (*Unsupervised Learning, Generative Learning and More*). Its goals are:

- to see the *curse of dimensionality* numerically (PC4),
- to compute a principal component analysis and to understand it as a reconstruction method,
- to compare nonlinear dimension reduction methods (MDS, Isomap, t-SNE, UMAP) on the digits and on a Swiss roll,
- to compare clustering methods (k-means, Gaussian mixtures, DBSCAN, agglomerative clustering) and to choose the number of groups,
- to generate new digits with a Gaussian mixture, and to train an autoencoder with `{torch}`.

The optional *To go further* part, after the core, can be skipped at first: it covers the rest of the lecture, in particular the generative models (VAE, flows, diffusion and flow matching, GAN).

::: {.no-handout}
We use `R` with `tidyverse`.

**Working environment.** Colab is the simplest option: open the notebook version of this lab (`APM_50441_EP_Lab_4.ipynb`, *Open in Colab* link) with the R runtime, and run its first cell, which installs the packages. To work on your own computer, download the lab folder (the `.qmd` file and its files) or clone the course repository, open the `.qmd` file in RStudio or Positron (or VS Code with the R extension), and run it chunk by chunk or render it with Quarto. The packages are installed once with the script `PC/env/install_packages.R` (`source("install_packages.R")`, then `install_lab("lab4")`); `{torch}` downloads its own library the first time (`torch::install_torch()`, done by `install_lab`). The data are read from the local files when they are present (otherwise they are downloaded).

**Use of AI.** AI assistants are allowed, but you must be able to explain every line you use, and you should answer the *predict before you run* questions before running the code: the suggested code may not be correct!
:::

In [ ]:
library(cluster)
library(igraph)
library(dbscan)
library(mclust)
library(Rtsne)
library(uwot)
library(torch)
library(patchwork)
library(tidyverse)   # loaded last, so that purrr::map is not masked by mclust::map
theme_set(theme_light())
if (!file.exists("data/digits.csv")) {
  dir.create("data", showWarnings = FALSE)
  download.file("https://raw.githubusercontent.com/APM-50441-EP/APM_50441_EP-2026-2027/main/PC4/data/digits.csv", "data/digits.csv")
}
digits <- read_csv("data/digits.csv", show_col_types = FALSE)
pixels <- as.matrix(select(digits, -label))
labels <- factor(digits$label)
dim(pixels)

## 1. The curse of dimensionality

PC4 shows that, for $n$ points uniform in $[0,1]^d$, the norms and the pairwise distances concentrate: $\|X\|/\sqrt{d} \to \sqrt{1/3}$ and $\|X - X'\|/\sqrt{d} \to \sqrt{1/6}$, and the ratio between the largest and the smallest distances tends to $1$ as long as $\log(n)/d \to 0$.

**Question** *Predict before you run:* for $250$ points uniform in $[0,1]^d$, what happens to the ratio between the largest and the smallest pairwise distances when $d$ grows? Compute, for $d \in \{2, 10, 100, 500, 1000\}$, the minimum, the maximum and the ratio of the distances to the origin and of the pairwise distances (with `dist()`), divided by $\sqrt{d}$.

**Answer**

In [ ]:
# Your code here

---

**Question** In $[0,1]^d$, a cube that contains a fraction $r$ of the data (uniform) has a side $r^{1/d}$. Plot this side for $r = 1\%$ and $r = 10\%$ as a function of $d \in [1, 100]$. What does it mean for a "local" method such as the $k$-nearest neighbors?

**Answer**

In [ ]:
# Your code here

---

**Question** Random projections do not suffer from the dimension: by the Johnson–Lindenstrauss lemma, projecting $n$ points on $k = O(\log(n)/\varepsilon^2)$ random directions preserves all their pairwise distances up to a factor $1 \pm \varepsilon$. Project $250$ Gaussian points of dimension $1000$ on $k \in \{10, 50, 200\}$ random Gaussian directions (scaled by $1/\sqrt{k}$), and plot the distribution of the ratios between the projected and the original pairwise distances.

**Answer**

In [ ]:
# Your code here

---

## 2. Principal component analysis of the digits

The digits are $8 \times 8$ grayscale images (values from $0$ to $16$): $1797$ observations in dimension $64$. A PCA looks for the linear subspace of dimension $q$ that keeps the largest inertia, or equivalently that reconstructs the data with the smallest error (PC4); it is computed by a singular value decomposition of the centered data matrix.

**Question** Compute the PCA of the digits with `prcomp` (without scaling), and check that its scores are the left singular vectors of the centered matrix multiplied by the singular values (`svd`), and that classical multidimensional scaling (`cmdscale`) of the Euclidean distances gives the same coordinates (up to signs). Plot the proportion of explained variance and the first two scores colored by the digit.

**Answer**

In [ ]:
# Your code here

---

**Question** Plot the mean digit and the first $6$ principal directions as $8\times 8$ images ("eigen-digits"). Then plot the reconstruction of the first $5$ digits with $q \in \{2, 5, 10, 20\}$ components, and the mean reconstruction error as a function of $q$. Relate this error to the explained variance.

**Answer**

In [ ]:
# Your code here

---

## 3. Nonlinear dimension reduction

A PCA can only find linear structures. On a *Swiss roll*, a 2D sheet rolled in 3D, the interesting coordinate is the position along the roll, which is not linear. *Isomap* replaces the Euclidean distances by *geodesic* distances (shortest paths in a neighborhood graph) and applies MDS to them; *t-SNE* and *UMAP* preserve the neighborhoods rather than the distances.

In [ ]:
set.seed(3)
t_roll <- 1.5 * pi * (1 + 2 * runif(1500))
swiss <- cbind(t_roll * cos(t_roll), 20 * runif(1500), t_roll * sin(t_roll))

**Question** Implement Isomap: build the graph of the $10$ nearest neighbors (`dbscan::kNN`), compute the shortest path distances (`igraph::distances`) and apply `cmdscale`. Compare the PCA and the Isomap embeddings of the Swiss roll, colored by the position along the roll `t_roll`.

**Answer**

In [ ]:
# Your code here

---

To compare embeddings, we use the proportion of the $10$ nearest neighbors of each point (in the original space) that are still among its $10$ nearest neighbors in the embedding:

In [ ]:
knn_preservation <- function(X, embedding, k = 10) {
  original <- kNN(X, k = k)$id
  embedded <- kNN(embedding, k = k)$id
  mean(map_dbl(seq_len(nrow(X)), \(i) length(intersect(original[i, ], embedded[i, ])) / k))
}

**Question** Embed the digits in 2D with PCA, Isomap, t-SNE (`Rtsne`, on the first $50$ principal components, perplexity $30$) and UMAP (`uwot::umap`). Plot the embeddings colored by the digit, and compare their neighbor preservation. Then try the perplexities $5$ and $100$ for t-SNE.

**Answer**

In [ ]:
# Your code here

---

## 4. Clustering

We compare the clustering methods of the lecture on three toy datasets with known groups, and on the digits. The agreement between a clustering and the true groups is measured with the *adjusted Rand index* (ARI): the proportion of pairs of points on which the two partitions agree (both in the same group, or both in different groups), corrected so that a random partition gets $0$ on average and identical partitions get $1$ (`mclust::adjustedRandIndex`).

In [ ]:
set.seed(5)
toy <- list(
  blobs = tibble(x = c(rnorm(200, 0), rnorm(200, 4), rnorm(200, 2)), y = c(rnorm(200, 0), rnorm(200, 0), rnorm(200, 3.5)), group = rep(1:3, each = 200)),
  elongated = tibble(x = c(rnorm(300, 0, 3), rnorm(300, 0, 3)), y = c(rnorm(300, 0, 0.3), rnorm(300, 2, 0.3)), group = rep(1:2, each = 300)),
  rings = tibble(angle = runif(600, 0, 2 * pi), radius = rep(c(1, 3), each = 300) + rnorm(600, 0, 0.15), group = rep(1:2, each = 300)) |>
    transmute(x = radius * cos(angle), y = radius * sin(angle), group))
bind_rows(toy, .id = "dataset") |> ggplot(aes(x, y, color = factor(group))) + geom_point(size = 0.5) +
  facet_wrap(~ dataset, scales = "free") + theme(aspect.ratio = 1) + labs(color = "group")

**Question** On each toy dataset, run $k$-means (`nstart = 20`), a Gaussian mixture (`Mclust`, with the true number of groups), DBSCAN (`dbscan`, $\varepsilon = 0.5$, `minPts = 10`; its noise points form a group of their own) and agglomerative clustering with the single, average and Ward linkages (`hclust` + `cutree`, true number of groups). Report the ARI of each method. Which assumptions explain the results?

**Answer**

In [ ]:
# Your code here

---

**Question** On the first $10$ principal components of the digits, choose the number of groups $K \in \{2, \dots, 15\}$ with the *elbow* of the within-cluster inertia of $k$-means, the average *silhouette* of $k$-means, and the BIC of Gaussian mixtures with diagonal covariances (`Mclust(..., modelNames = "VVI")`). Compare the chosen numbers with the $10$ digits, and the ARI of the clusterings with the labels.

**Answer**

In [ ]:
# Your code here

---

## 5. A first generative model: Gaussian mixtures

A density model can be used to *generate* new data: a Gaussian mixture is sampled by drawing a component, then a Gaussian vector. In dimension $64$, the covariances of a mixture are hard to estimate, so we fit the mixture on the first principal components and map the samples back to the pixels.

**Question** Fit a Gaussian mixture with $12$ components and full covariances (`Mclust(..., G = 12, modelNames = "VVV")`) on the first $15$ principal components of the digits, draw $30$ new points with `mclust::sim`, map them back to the pixel space and plot them. Compare with a mixture on $5$ components only.

**Answer**

In [ ]:
# Your code here

---

## 6. Autoencoders with `{torch}`

An autoencoder is a neural network trained to reconstruct its input through a low-dimensional *code*: an encoder $x \mapsto z$ and a decoder $z \mapsto \hat x$, trained to minimize $\|x - \hat x\|^2$. With linear layers and the squared loss, the best autoencoder spans the same subspace as the PCA; nonlinear layers can find curved structures.

In [ ]:
torch_set_num_threads(2)                            # small networks: a few threads are faster than many
X_scaled <- scale(pixels, scale = FALSE) / 16       # centered, in [-1, 1]
x_tensor <- torch_tensor(X_scaled, dtype = torch_float())
train_autoencoder <- function(encoder, decoder, epochs = 1500, lr = 1e-2, seed = 0) {
  torch_manual_seed(seed)
  parameters <- c(encoder$parameters, decoder$parameters)
  optimizer <- optim_adam(parameters, lr = lr)
  losses <- numeric(epochs)
  for (epoch in seq_len(epochs)) {
    optimizer$zero_grad()
    loss <- nnf_mse_loss(decoder(encoder(x_tensor)), x_tensor)
    loss$backward()
    optimizer$step()
    losses[epoch] <- loss$item()
  }
  losses
}

**Question** Train a *linear* autoencoder $64 \to 2 \to 64$ (`nn_linear` layers, full batch, Adam, $1500$ epochs), and compare its reconstruction error with the PCA with $2$ components. Check that the subspace spanned by the decoder weights is the one of the first two principal directions, by computing the cosines of the *principal angles* between the two subspaces (the singular values of $Q_1^\top Q_2$, where $Q_1$ and $Q_2$ are orthonormal bases).

**Answer**

In [ ]:
# Your code here

---

**Question** Train a deep autoencoder $64 \to 128 \to 32 \to 2 \to 32 \to 128 \to 64$ with `tanh` activations. Compare its reconstruction error with the linear one, and plot the 2D codes of both, colored by the digit.

**Answer**

In [ ]:
# Your code here

---

## About this lab

**Use of AI.** Artificial intelligence tools have been used to revise the content of this lab. The main contributor takes full responsibility for this content.

**License.** The texts of this lab are under the [Creative Commons Attribution-ShareAlike 4.0](https://creativecommons.org/licenses/by-sa/4.0/) license (CC BY-SA 4.0), and its code under the [MIT](https://opensource.org/license/mit) license; the datasets keep their own licenses. Main contributor: E. Le Pennec. Contributors: S. Boucheron, A. Dieuleveut, A.K. Fermin, S. Gadat, S. Gaiffas, B. Goujaud, A. Guilloux, Ch. Keribin, Th. Klein, E. Matzner, J. Read, M. Sangnier, E. Scornet.